In [16]:
import pandas as pd
from pathlib import Path 
import numpy as np

In [17]:
orders710 = pd.read_csv('../../orderlevel_updated/tier_merged/tiers_merged_0710-0713.csv')

In [18]:
orders710.head(5)

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,total_user_waiting_time_second,order_status,date,region::multi-filter,avg_rain,Rain_Level,Rider Group,Start Date,End Date,Incentive
0,0,SURIN,LMD00BZBZ,2025-07-10 10:27:54.432,ก๋วยเตี๋ยวถนอมมิตร2 (Thanommit Noodles 2),Noodles,LMF-250710-408959062,True,13.50,3.0,...,687,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,No Incentive
1,1,SURIN,LMD00BZBZ,2025-07-10 10:32:29.887,ตำซี๊ด&อาหารตามสั่ง (Tum Zeed & Made-to-Order ...,Dessert,LMF-250710-161870697,True,17.80,3.0,...,1223,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,No Incentive
2,2,SURIN,LMD00BZBZ,2025-07-10 10:47:04.631,เฉิงกง ติ่มซำ (Chenggong Dim Sum) -,Dim Sum,LMF-250710-166811972,True,12.66,3.0,...,2458,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,No Incentive
3,3,SURIN,LMD00BZBZ,2025-07-10 10:56:22.791,Café Amazon - SD3192 (คาเฟ่ อเมซอน - SD3192) ส...,Café/Coffee Shop,LMF-250710-400808907,True,12.37,3.0,...,1180,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,No Incentive
4,4,SURIN,LMD00BZBZ,2025-07-10 11:09:04.244,Café Amazon - DD1655 (คาเฟ่ อเมซอน - DD1655) ป...,Café/Coffee Shop,LMF-250710-858627724,True,14.30,3.0,...,2035,COMPLETED,2025-07-10,SURIN,0.0,no-rain,Mid,2025-07-10,2025-07-13,No Incentive


In [19]:
orders710.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 446844 entries, 0 to 446843
Data columns (total 27 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Unnamed: 0                                  446844 non-null  int64  
 1   region                                      446844 non-null  object 
 2   driver_id                                   446844 non-null  object 
 3   assign_created_at                           446844 non-null  object 
 4   restaurant_name                             446844 non-null  object 
 5   restaurant_category                         446844 non-null  object 
 6   order_id                                    446844 non-null  object 
 7   is_driver_accept                            446844 non-null  bool   
 8   wage                                        446844 non-null  float64
 9   on_top_fare                                 446844 non-null  float64
 

In [20]:
# Setup paths
csv_files = sorted(Path('../daily_guarantee/daily_guarantee_results').glob('*.csv'))
output_dir = Path('./active_hour_results')
output_dir.mkdir(parents=True, exist_ok=True)

# Display all files
print("Files to process:")
for i, file in enumerate(csv_files, 1):
    print(f"{i}. {file.name}")

all_dfs = []
for i, csv_file in enumerate(csv_files, 1):
    print(f"[{i}/{len(csv_files)}] Loading: {csv_file.name}")
    df_temp = pd.read_csv(csv_file)
    all_dfs.append(df_temp)

# Combine all datasets
df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal records after merging: {len(df):,}")

Files to process:
1. daily_guarantee_daily_productivity_tiers_merged_0710-0713.csv
2. daily_guarantee_daily_productivity_tiers_merged_0714-0716.csv
3. daily_guarantee_daily_productivity_tiers_merged_0717-0720.csv
4. daily_guarantee_daily_productivity_tiers_merged_0721-0725.csv
5. daily_guarantee_daily_productivity_tiers_merged_0726-0727.csv
6. daily_guarantee_daily_productivity_tiers_merged_0728-0801.csv
7. daily_guarantee_daily_productivity_tiers_merged_0802-0806.csv
8. daily_guarantee_daily_productivity_tiers_merged_0807-0810.csv
9. daily_guarantee_daily_productivity_tiers_merged_0811-0815.csv
10. daily_guarantee_daily_productivity_tiers_merged_0816-0820.csv
11. daily_guarantee_daily_productivity_tiers_merged_0821-0825.csv
12. daily_guarantee_daily_productivity_tiers_merged_0826-0829.csv
[1/12] Loading: daily_guarantee_daily_productivity_tiers_merged_0710-0713.csv
[2/12] Loading: daily_guarantee_daily_productivity_tiers_merged_0714-0716.csv
[3/12] Loading: daily_guarantee_daily_produ

In [21]:

# Convert datetime
df['assign_created_at'] = pd.to_datetime(df['assign_created_at'])
df = df.sort_values(['driver_id', 'assign_created_at']).reset_index(drop=True)

# Ensure numeric columns are properly typed
df['rider_waiting_time_in_restaurant_second'] = pd.to_numeric(df['rider_waiting_time_in_restaurant_second'], errors='coerce').fillna(0)
df['delivery_time_to_user_second'] = pd.to_numeric(df['delivery_time_to_user_second'], errors='coerce').fillna(0)

# Filter only accepted orders
accepted_mask = df['is_driver_accept'] == True


In [22]:

# Time per order
df['time_per_order_sec'] = df['rider_waiting_time_in_restaurant_second'] + df['delivery_time_to_user_second']

# Total seconds worked each day (0 if not accepted)
df['time_worked_sec'] = np.where(accepted_mask, df['time_per_order_sec'], 0)
df['time_worked_sec'] = df['time_worked_sec'].astype(float)

# Extract date, hour, and day of week
df['date'] = df['assign_created_at'].dt.date
df['hour'] = df['assign_created_at'].dt.hour
df['day_of_week'] = df['assign_created_at'].dt.day_name()
df['day_of_week_short'] = df['assign_created_at'].dt.strftime('%a')  # Mon, Fri, Sat, Sun format


In [23]:

# Calculate week number where week starts on Monday
df['week_start_date'] = (df['assign_created_at'] - pd.to_timedelta(df['assign_created_at'].dt.dayofweek, unit='D')).dt.date

# For grouping, create a week identifier
df['year'] = df['assign_created_at'].dt.year
df['week'] = df['assign_created_at'].dt.isocalendar().week  # ISO week already starts on Monday


In [24]:

# Cumulative sum per driver per date
df['daily_cumulative_time_sec'] = df.groupby(['driver_id', 'date'])['time_worked_sec'].cumsum()

# Masks for time windows (10:00-12:59 or 17:00-19:59)
time_window_mask = ((df['hour'] >= 10) & (df['hour'] <= 12)) | ((df['hour'] >= 17) & (df['hour'] <= 19))

# Calculate time worked for orders in the window
df['time_worked_in_window_sec'] = np.where(time_window_mask, df['time_worked_sec'], 0)
df['time_worked_in_window_sec'] = df['time_worked_in_window_sec'].astype(float)

# Running sum within each driver per day
df['window_cumulative_sec'] = df.groupby(['driver_id', 'date'])['time_worked_in_window_sec'].cumsum()

# Convert to hours
df['daily_cumulative_time_hr'] = df['daily_cumulative_time_sec'] / 3600
df['window_cumulative_hr'] = df['window_cumulative_sec'] / 3600


In [25]:

# Get unique dates per driver per week (for running count)
# Group by week_start_date to ensure Monday-based weeks
daily_summary = df.groupby(['driver_id', 'week_start_date', 'date']).agg({
    'day_of_week_short': 'first',
    'year': 'first',
    'week': 'first'
}).reset_index()

# Sort by driver, week_start_date, and date
daily_summary = daily_summary.sort_values(['driver_id', 'week_start_date', 'date'])

# Create running count of unique days worked so far in the week
# automatically resets for each new (driver_id, week_start_date) group
daily_summary['days_worked_count'] = daily_summary.groupby(['driver_id', 'week_start_date']).cumcount() + 1

# Create running set of days worked so far in the week
required_days = {'Mon', 'Fri', 'Sat', 'Sun'}

def get_running_days_info(group):
    running_lists = []
    running_has_required = []
    current_set = set()
    
    for day in group['day_of_week_short'].values:
        current_set.add(day)
        running_lists.append(', '.join(sorted(current_set)))
        running_has_required.append(required_days.issubset(current_set))
    
    return pd.DataFrame({
        'driver_id': group['driver_id'].values,
        'week_start_date': group['week_start_date'].values,
        'date': group['date'].values,
        'year': group['year'].values,
        'week': group['week'].values,
        'days_worked_count': group['days_worked_count'].values,
        'days_worked_list': running_lists,
        'has_required_days': running_has_required
    })

# This processes each (driver_id, week_start_date) group independently
# So it automatically resets every Monday
daily_summary = daily_summary.groupby(['driver_id', 'week_start_date'], group_keys=False).apply(
    get_running_days_info
).reset_index(drop=True)

/var/folders/s_/0td42yfs6wjd9wdpc5nr90400000gn/T/ipykernel_52584/2681475837.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_summary = daily_summary.groupby(['driver_id', 'week_start_date'], group_keys=False).apply(


In [26]:

#merge to og dataset
df = df.merge(
    daily_summary[['driver_id', 'week_start_date', 'date', 'days_worked_count', 
                   'days_worked_list', 'has_required_days']],
    on=['driver_id', 'week_start_date', 'date'],
    how='left'
)

print("Merged metrics back to main dataframe...")

#save
output_path = output_dir / 'active_hour_merged.csv'
df.to_csv(output_path, index=False)

print(f"\n{'='*80}")
print("Processing complete!")
print(f"{'='*80}\n")

Merged metrics back to main dataframe...

Processing complete!



In [27]:
active_hour = pd.read_csv('./active_hour_results/active_hour.csv')

In [28]:
active_hour.head(10)

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,window_cumulative_hr,Unnamed: 0.1,day_of_week,day_of_week_short,week_start_date,year,week,days_worked_count,days_worked_list,has_required_days
0,0,SURIN,LMD00BZBZ,2025-07-10 10:27:54.432,ก๋วยเตี๋ยวถนอมมิตร2 (Thanommit Noodles 2),Noodles,LMF-250710-408959062,True,13.50,3.0,...,0.083611,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
1,1,SURIN,LMD00BZBZ,2025-07-10 10:32:29.887,ตำซี๊ด&อาหารตามสั่ง (Tum Zeed & Made-to-Order ...,Dessert,LMF-250710-161870697,True,17.80,3.0,...,0.195278,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
2,2,SURIN,LMD00BZBZ,2025-07-10 10:47:04.631,เฉิงกง ติ่มซำ (Chenggong Dim Sum) -,Dim Sum,LMF-250710-166811972,True,12.66,3.0,...,0.620556,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
3,3,SURIN,LMD00BZBZ,2025-07-10 10:56:22.791,Café Amazon - SD3192 (คาเฟ่ อเมซอน - SD3192) ส...,Café/Coffee Shop,LMF-250710-400808907,True,12.37,3.0,...,0.670556,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
4,4,SURIN,LMD00BZBZ,2025-07-10 11:09:04.244,Café Amazon - DD1655 (คาเฟ่ อเมซอน - DD1655) ป...,Café/Coffee Shop,LMF-250710-858627724,True,14.30,3.0,...,0.845556,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
5,5,SURIN,LMD00BZBZ,2025-07-10 11:11:38.237,Café Amazon - SD3192 (คาเฟ่ อเมซอน - SD3192) ส...,Café/Coffee Shop,LMF-250710-321972404,True,12.37,3.0,...,0.966111,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
6,6,SURIN,LMD00BZBZ,2025-07-10 11:23:47.869,Café Amazon - DD1655 (คาเฟ่ อเมซอน - DD1655) ป...,Café/Coffee Shop,LMF-250710-771935451,True,14.30,3.0,...,1.181667,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
7,7,SURIN,LMD00BZBZ,2025-07-10 11:33:30.396,PARLOR,Cafe,LMF-250710-015980523,True,11.40,3.0,...,1.566667,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
8,8,SURIN,LMD00BZBZ,2025-07-10 11:40:19.467,PunThai Coffee (กาแฟพันธุ์ไทย) ถนนหลักเมือง สุ...,Café/Coffee Shop,LMF-250710-466168679,True,11.40,3.0,...,1.989444,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False
9,9,SURIN,LMD00BZBZ,2025-07-10 11:49:49.241,ลุงแหลมอาหารตามสั่งหน้าเทคนิค (Uncle Laem's Ma...,À La Carte,LMF-250710-422617314,True,11.40,3.0,...,2.156111,NaN,Thursday,Thu,2025-07-07,2025,28,1,Thu,False


In [29]:
LMD00BZBZ = active_hour[active_hour['driver_id'] == 'LMD00BZBZ']

In [30]:
LMD00BZBZ[LMD00BZBZ['week'] == 32]

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,window_cumulative_hr,Unnamed: 0.1,day_of_week,day_of_week_short,week_start_date,year,week,days_worked_count,days_worked_list,has_required_days
550,79,SURIN,LMD00BZBZ,2025-08-04 07:59:47.037,Follow the sun Follow the sun,Café/Coffee Shop,LMF-250804-560867621,True,11.0,3.0,...,0.000000,NaN,Monday,Mon,2025-08-04,2025,32,1,Mon,False
551,80,SURIN,LMD00BZBZ,2025-08-04 08:03:09.875,สดกาแฟ (กาแฟสด) (SodCoffee),Café/Coffee Shop,LMF-250804-075667516,True,11.0,3.0,...,0.000000,NaN,Monday,Mon,2025-08-04,2025,32,1,Mon,False
552,81,SURIN,LMD00BZBZ,2025-08-04 08:04:13.564,Café Amazon - SC1631 (คาเฟ่ อเมซอน - SC1631) โ...,Café/Coffee Shop,LMF-250804-449082737,True,11.0,3.0,...,0.000000,NaN,Monday,Mon,2025-08-04,2025,32,1,Mon,False
553,82,SURIN,LMD00BZBZ,2025-08-04 08:29:12.059,เจ๊มุ้ยหมูปิ้ง หน้าซอยโดนไข1 - ในเมือง (Jae Mu...,Breakfast/Brunch,LMF-250804-304361482,True,13.7,3.0,...,0.000000,NaN,Monday,Mon,2025-08-04,2025,32,1,Mon,False
554,83,SURIN,LMD00BZBZ,2025-08-04 08:34:11.987,ข้าวมันไก่ภัสสร PASSORN หน้า สวค.,Thai,LMF-250804-681603399,True,13.7,3.0,...,0.000000,NaN,Monday,Mon,2025-08-04,2025,32,1,Mon,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1664,988,SURIN,LMD00BZBZ,2025-08-10 21:22:27.898,ยำแซ่บนัวร์(Yumzapnuar)-หลังมอ (Yum Zap Nuar),Street Food/Food Stands,LMF-250810-117366282,False,17.0,3.0,...,32.741667,988.0,Sunday,Sun,2025-08-04,2025,32,7,"Fri, Mon, Sat, Sun, Thu, Tue, Wed",True
1665,987,SURIN,LMD00BZBZ,2025-08-10 21:22:27.898,ยำแซ่บนัวร์(Yumzapnuar)-หลังมอ (Yum Zap Nuar),Street Food/Food Stands,LMF-250810-117366282,False,17.0,3.0,...,32.741667,987.0,Sunday,Sun,2025-08-04,2025,32,7,"Fri, Mon, Sat, Sun, Thu, Tue, Wed",True
1666,986,SURIN,LMD00BZBZ,2025-08-10 21:22:27.898,ยำแซ่บนัวร์(Yumzapnuar)-หลังมอ (Yum Zap Nuar),Street Food/Food Stands,LMF-250810-117366282,False,17.0,3.0,...,32.741667,986.0,Sunday,Sun,2025-08-04,2025,32,7,"Fri, Mon, Sat, Sun, Thu, Tue, Wed",True
1667,985,SURIN,LMD00BZBZ,2025-08-10 21:22:27.898,ยำแซ่บนัวร์(Yumzapnuar)-หลังมอ (Yum Zap Nuar),Street Food/Food Stands,LMF-250810-117366282,False,17.0,3.0,...,32.741667,985.0,Sunday,Sun,2025-08-04,2025,32,7,"Fri, Mon, Sat, Sun, Thu, Tue, Wed",True
